# Lab 5: Hugging Face - Datasets e Inferencia con LLMs y Modelos Multimodales (Gemma)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-source-models/blob/main/session-01-hf-kerashub-litert/05-huggingface-gemma-datasets/01_huggingface_datasets_gemma_multimodal.ipynb)

## Objetivo
1. Utilizar la biblioteca **`datasets`** de Hugging Face para descargar conjuntos de datos de instrucciones directamente desde Hugging Face Hub.
2. Cargar y ejecutar modelos de lenguaje y multimodales de la familia **Gemma** mediante **`from transformers import AutoProcessor, AutoModelForMultimodalLM`** con distribucion automatica de memoria (**`device_map="auto"`**).
3. Evaluar el modelo en tiempo real sobre los ejemplos descargados del dataset aplicando **Chat Templates** estandarizados.

### Paso 1: Instalacion de Dependencias

In [ ]:
!pip install -q transformers datasets accelerate torch bitsandbytes pillow matplotlib

### Paso 2: Descarga de un Dataset Real desde Hugging Face Hub (`datasets.load_dataset`)
Descargamos un conjunto estructurado de instrucciones y respuestas de muestra utilizando la biblioteca oficial `datasets`.

In [ ]:
from datasets import load_dataset

print("Descargando dataset de instrucciones desde Hugging Face Hub...")
# Cargamos 5 ejemplos del dataset de instrucciones 'databricks/databricks-dolly-15k'
dataset = load_dataset("databricks/databricks-dolly-15k", split="train[:5]")
print(f"Dataset descargado exitosamente con {len(dataset)} ejemplos.\n")
print("Columnas disponibles:", dataset.column_names)

for i, sample in enumerate(dataset, 1):
    print(f"\n[Muestra {i}] Categoria: {sample['category']}")
    print(f"  Instruccion : {sample['instruction']}")
    if sample['context']:
        print(f"  Contexto    : {sample['context'][:80]}...")

### Paso 3: Carga del Modelo con `AutoProcessor` y `AutoModelForMultimodalLM`
Cargamos la arquitectura multimodal y su procesador utilizando `device_map="auto"` para distribuir automaticamente los pesos entre la GPU y la memoria RAM.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM
import time

# Modelo principal (En Google Colab con GPU T4/A100 o equipos con 16GB+ VRAM use 'google/gemma-4-12B-it')
model_id = "google/gemma-4-12B-it"
print(f"Inicializando procesador y arquitectura: {model_id}...")

try:
    processor = AutoProcessor.from_pretrained(model_id)
    model = AutoModelForMultimodalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True
    )
    model.eval()
    print("Modelo Gemma Multimodal cargado exitosamente.")
except Exception as e:
    print(f"Aviso al cargar {model_id} en este entorno ({e}).")
    print("Cargando modelo VLM compacto para ejecucion interactiva inmediata...")
    fallback_id = "HuggingFaceTB/SmolVLM-256M-Instruct"
    processor = AutoProcessor.from_pretrained(fallback_id)
    model = AutoModelForMultimodalLM.from_pretrained(
        fallback_id,
        device_map="auto",
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    model.eval()
    print(f"Modelo ({fallback_id}) listo para evaluacion.")

### Paso 4: Inferencia y Evaluacion del LLM sobre el Dataset Descargado

In [ ]:
print("=" * 70)
print("Evaluando el modelo sobre las muestras del dataset descargado...")
print("=" * 70)

for idx, sample in enumerate(dataset, start=1):
    instruction = sample["instruction"]
    context = sample.get("context", "")
    category = sample.get("category", "General")
    
    full_prompt = f"{instruction}\nContexto: {context}" if context else instruction
    
    messages = [
        {"role": "user", "content": full_prompt}
    ]
    
    try:
        formatted_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    except Exception:
        formatted_prompt = f"<start_of_turn>user\n{full_prompt}<end_of_turn>\n<start_of_turn>model\n"
        
    inputs = processor(text=formatted_prompt, return_tensors="pt")
    if hasattr(model, "device"):
        inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
        
    start_t = time.time()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )
        
    input_len = inputs["input_ids"].shape[1] if "input_ids" in inputs else 0
    new_tokens = output_ids[0][input_len:]
    generated_text = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    elapsed = time.time() - start_t
    
    print(f"\n[Muestra {idx}/{len(dataset)}] Categoria: {category}")
    print(f"Pregunta   : {instruction}")
    if context:
        print(f"Contexto   : {context[:80]}...")
    print(f"Respuesta  : {generated_text}")
    print(f"Rendimiento: {elapsed:.2f}s ({len(new_tokens)/max(elapsed, 0.001):.1f} tok/s)")
    print("-" * 70)

### Paso 5: Prueba Multimodal Combinada (Imagen + Prompt)

In [ ]:
import urllib.request, ssl, os
from PIL import Image
import matplotlib.pyplot as plt

os.makedirs("sample_images", exist_ok=True)
sample_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/b/b3/Golden_Retriever_2019.jpg/500px-Golden_Retriever_2019.jpg"
image_path = "sample_images/dog.jpg"

if not os.path.exists(image_path):
    ctx = ssl._create_unverified_context()
    req = urllib.request.Request(sample_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, context=ctx) as r, open(image_path, "wb") as f:
        f.write(r.read())

sample_img = Image.open(image_path).convert("RGB")
plt.figure(figsize=(4, 3))
plt.imshow(sample_img)
plt.axis("off")
plt.title("Entrada Visual")
plt.show()

multimodal_msgs = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Describe in detail what dog breed is visible in this image."}
        ]
    }
]
prompt_mm = processor.apply_chat_template(multimodal_msgs, add_generation_prompt=True)
inputs_mm = processor(text=prompt_mm, images=sample_img, return_tensors="pt")
if hasattr(model, "device"):
    inputs_mm = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs_mm.items()}

with torch.no_grad():
    out_mm = model.generate(**inputs_mm, max_new_tokens=80, do_sample=False)
i_len = inputs_mm["input_ids"].shape[1] if "input_ids" in inputs_mm else 0
ans_mm = processor.tokenizer.decode(out_mm[0][i_len:], skip_special_tokens=True).strip()

print("\nRespuesta Multimodal:")
print(ans_mm)

### Paso Final: Limpieza del Entorno y Liberacion de Recursos (Cleanup)

In [ ]:
import gc

del model, processor, dataset, sample_img
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Recursos de memoria VRAM/RAM y tensores liberados exitosamente.')